# RQ1 Java — análise e gráficos

Este notebook analisa o JSON produzido por `python -m src.java_rq1`. Execute-o a partir da pasta `pipeline` ou ajuste `JAVA_RESULTS_PATH` na primeira célula de código.

Os gráficos usam os clones Java aprovados que foram mantidos como representantes após o clustering. O arquivo atual não registra tentativas rejeitadas, resultados de reprompt nem tamanho original de cada cluster; por isso, as seções abaixo deixam explícitos os gráficos da RQ1 Python que não podem ser reproduzidos de forma equivalente.

In [ ]:
import json
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt

# Altere esta variável se o resultado estiver em outro local.
JAVA_RESULTS_PATH = None
DEFAULT_RESULT_NAME = "human_eval_clone_dataset.json"

if JAVA_RESULTS_PATH is not None:
    results_path = Path(JAVA_RESULTS_PATH).expanduser()
else:
    candidates = [
        Path("../results/RQ1") / DEFAULT_RESULT_NAME,
        Path("results/RQ1") / DEFAULT_RESULT_NAME,
        Path("pipeline/results/RQ1") / DEFAULT_RESULT_NAME,
    ]
    results_path = next((path for path in candidates if path.exists()), candidates[0])

if not results_path.is_file():
    raise FileNotFoundError(
        f"Não encontrei o resultado Java em {results_path}. "
        "Execute `python -m src.java_rq1` na pasta pipeline ou configure JAVA_RESULTS_PATH."
    )

with results_path.open("r", encoding="utf-8") as result_file:
    dataset = json.load(result_file)

entries = [entry for entry in dataset if entry.get("language") == "java"]
clones = [clone for entry in entries for clone in entry.get("clones", [])]

print(f"Arquivo: {results_path.resolve()}")
print(f"Entradas Java: {len(entries)}")
print(f"Representantes/clones salvos: {len(clones)}")
print(f"Modelos presentes: {sorted({clone.get('model', 'unknown') for clone in clones})}")

try:
    from codebleu import calc_codebleu
    CODEBLEU_AVAILABLE = True
except ImportError as error:
    calc_codebleu = None
    CODEBLEU_AVAILABLE = False
    print(f"CodeBLEU indisponível; a seção de similaridade será ignorada: {error}")

In [ ]:
def refac_label(clone):
    refacs = clone.get("refacs", [])
    if isinstance(refacs, str):
        return refacs or "no-refac"
    return "+".join(refacs) if refacs else "no-refac"


def clones_for(entry):
    return entry.get("clones", [])


def plot_grouped_bars(ax, grouped, title, ylabel):
    labels = sorted(grouped)
    if not labels:
        ax.text(0.5, 0.5, "Sem dados", ha="center", va="center")
        ax.set_axis_off()
        return

    models = sorted({model for values in grouped.values() for model in values})
    width = 0.8 / max(len(models), 1)
    positions = range(len(labels))
    for model_index, model in enumerate(models):
        values = [grouped[label].get(model, 0) for label in labels]
        offsets = [x - 0.4 + (model_index + 0.5) * width for x in positions]
        ax.bar(offsets, values, width=width, label=model)

    ax.set_xticks(list(positions))
    ax.set_xticklabels(labels, rotation=35, ha="right")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()


print("Helpers de análise carregados.")

## Similaridade CodeBLEU entre clones Java

O gráfico original da RQ1 Python usa CodeBLEU **original-vs-clone**. O fluxo Java não grava esse score e o `original_code` Java exportado é o template com os testes, não uma implementação de referência completa. Portanto, esse score equivalente não pode ser calculado a partir do arquivo salvo.

A célula seguinte calcula uma medida diferente, mas disponível: a similaridade CodeBLEU Java média de cada clone em relação aos outros clones salvos da mesma entrada, e plota sua distribuição por combinação de refacs e modelo. O código Java comparado inclui o template/testes compartilhados, como no clustering do pipeline.

In [ ]:
peer_scores = defaultdict(list)

if not CODEBLEU_AVAILABLE:
    print("Gráfico de similaridade não gerado: instale as dependências de pipeline/requirements.txt.")
else:
    for entry in entries:
        entry_clones = clones_for(entry)
        per_clone_scores = [[] for _ in entry_clones]
        for left, right in combinations(range(len(entry_clones)), 2):
            score = float(
                calc_codebleu(
                    [entry_clones[left]["code"]],
                    [entry_clones[right]["code"]],
                    lang="java",
                )["codebleu"]
            )
            per_clone_scores[left].append(score)
            per_clone_scores[right].append(score)

        for clone, scores in zip(entry_clones, per_clone_scores):
            if scores:
                key = (refac_label(clone), clone.get("model", "unknown"))
                peer_scores[key].append(sum(scores) / len(scores))

    refac_groups = sorted({refacs for refacs, _ in peer_scores})
    models = sorted({model for _, model in peer_scores})
    if not peer_scores:
        print("Sem pares de clones da mesma entrada; não há similaridades para plotar.")
    else:
        fig, ax = plt.subplots(figsize=(max(9, len(refac_groups) * 1.5), 5))
        width = 0.8 / max(len(models), 1)
        for model_index, model in enumerate(models):
            positions = [
                group_index - 0.4 + (model_index + 0.5) * width
                for group_index, refacs in enumerate(refac_groups)
                if peer_scores.get((refacs, model))
            ]
            values = [
                peer_scores[(refacs, model)]
                for refacs in refac_groups
                if peer_scores.get((refacs, model))
            ]
            if values:
                boxes = ax.boxplot(
                    values,
                    positions=positions,
                    widths=width * 0.85,
                    patch_artist=True,
                    showfliers=False,
                    medianprops={"color": "black"},
                )
                for box in boxes["boxes"]:
                    box.set_alpha(0.65)
                boxes["boxes"][0].set_label(model)

        ax.set_xticks(range(len(refac_groups)))
        ax.set_xticklabels(refac_groups, rotation=35, ha="right")
        ax.set_ylabel("Média de CodeBLEU contra clones pares")
        ax.set_title("Similaridade entre representantes Java por modelo e refacs")
        ax.legend()
        ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.show()

## Resultados dos testes

Este gráfico espelha a divisão da RQ1 por contexto, estratégia e refacs. A taxa é calculada **somente sobre clones presentes no JSON Java**. Como o pipeline descarta candidatos que falham compilação/testes antes de salvar, a taxa tende a ser 100% e não representa a taxa de aprovação de todas as gerações.

In [ ]:
test_groups = {
    "context": defaultdict(lambda: defaultdict(lambda: [0, 0])),
    "strategy": defaultdict(lambda: defaultdict(lambda: [0, 0])),
    "refacs": defaultdict(lambda: defaultdict(lambda: [0, 0])),
}

for clone in clones:
    model = clone.get("model", "unknown")
    results = clone.get("test_results", {})
    if not results:
        continue
    factors = {
        "context": [clone.get("context", "unknown")],
        "strategy": [clone.get("strategy", "unknown")],
        "refacs": clone.get("refacs", []) or ["no-refac"],
    }
    for factor, labels in factors.items():
        for label in labels:
            for status in results.values():
                test_groups[factor][label][model][1] += 1
                if isinstance(status, str) and status.upper() == "PASS":
                    test_groups[factor][label][model][0] += 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, factor in zip(axes, ("context", "strategy", "refacs")):
    grouped = {}
    for label, model_values in test_groups[factor].items():
        grouped[label] = {
            model: (passed / total * 100 if total else 0)
            for model, (passed, total) in model_values.items()
        }
    plot_grouped_bars(ax, grouped, f"Testes aprovados por {factor}", "Aprovação (%)")
    ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

## Clones mantidos por configuração

O gráfico abaixo mostra contagens observadas no JSON, após o clustering: são representantes aprovados, não o total de tentativas nem todos os candidatos válidos anteriores à redução.

In [ ]:
count_by_model = defaultdict(Counter)
count_by_refacs = defaultdict(Counter)
for clone in clones:
    model = clone.get("model", "unknown")
    count_by_model["todos os clones"][model] += 1
    count_by_refacs[refac_label(clone)][model] += 1

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_grouped_bars(axes[0], count_by_model, "Representantes salvos por modelo", "Clones")
plot_grouped_bars(axes[1], count_by_refacs, "Representantes salvos por combinação de refacs", "Clones")
plt.tight_layout()
plt.show()

## Contagem de representantes por problema

O pipeline Java salva um representante por cluster e marca-o com `cluster`. O tamanho de cada cluster original não é gravado, então não é possível reproduzir estatísticas como clones por cluster; esta distribuição mostra apenas quantos representantes foram salvos para cada entrada.

In [ ]:
representatives_per_entry = {
    entry.get("id", "unknown"): len(clones_for(entry))
    for entry in entries
}
if representatives_per_entry:
    labels = list(representatives_per_entry)
    values = [representatives_per_entry[label] for label in labels]
    fig, ax = plt.subplots(figsize=(max(12, len(labels) * 0.22), 5))
    ax.bar(range(len(labels)), values, color="steelblue")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    ax.set_ylabel("Representantes salvos")
    ax.set_title("Representantes Java por problema")
    plt.tight_layout()
    plt.show()
else:
    print("Não há entradas Java no arquivo carregado.")

## Gráficos da RQ1 Python não reproduzidos

- **CodeBLEU original-vs-clone:** o JSON Java não contém uma implementação de referência Java nem esse score. A célula de similaridade acima é clone-vs-clone e não deve ser interpretada como o mesmo resultado.
- **Reprompt e reprompts falhos:** não existem etapas/arquivos de reprompt no fluxo Java atual.
- **Tamanho dos clusters:** só os representantes são salvos; os membros e as contagens originais dos clusters não são preservados.
- **Eficiência / survival ratio:** o fluxo não registra tentativas rejeitadas ou o número total de gerações por configuração, portanto não há denominador confiável para calcular a sobrevivência.

Para habilitar esses gráficos de forma fiel, o pipeline precisaria salvar o código original Java completo, CodeBLEU original-vs-clone, tentativas e motivos de rejeição, resultados dos membros antes do clustering e dados de reprompt, se essa etapa for adicionada.